# Recommender System

In [1]:
import numpy as np
from numpy.typing import NDArray
from typing import Tuple
import pandas as pd
from numpy import loadtxt
import time

import random

random.seed(42)

In [2]:
data_path = './data/recommender_system/'

def load_precalc_params_small():

    file = open(f'{data_path}/small_movies_X.csv', 'rb')
    X = loadtxt(file, delimiter = ",")

    file = open(f'{data_path}/small_movies_W.csv', 'rb')
    W = loadtxt(file,delimiter = ",")

    file = open(f'{data_path}/small_movies_b.csv', 'rb')
    b = loadtxt(file,delimiter = ",")
    b = b.reshape(1,-1)
    num_movies, num_features = X.shape
    num_users,_ = W.shape
    return(X, W, b, num_movies, num_features, num_users)
    
def load_ratings_small():
    file = open(f'{data_path}/small_movies_Y.csv', 'rb')
    Y = loadtxt(file,delimiter = ",")

    file = open(f'{data_path}/small_movies_R.csv', 'rb')
    R = loadtxt(file,delimiter = ",")
    return(Y,R)

def load_Movie_List_pd():
    """ returns df with and index of movies in the order they are in in the Y matrix """
    df = pd.read_csv(f'{data_path}/small_movie_list.csv', header=0, index_col=0,  delimiter=',', quotechar='"')
    mlist = df["title"].to_list()
    return(mlist, df)

def test_cofi_cost_func(target):
    num_users_r = 4
    num_movies_r = 5 
    num_features_r = 3

    X_r = np.ones((num_movies_r, num_features_r))
    W_r = np.ones((num_users_r, num_features_r))
    b_r = np.zeros((1, num_users_r))
    Y_r = np.zeros((num_movies_r, num_users_r))
    R_r = np.zeros((num_movies_r, num_users_r))
    
    J = target(X_r, W_r, b_r, Y_r, R_r, 2);
    assert not np.isclose(J, 13.5), f"Wrong value. Got {J}. Did you multiplied the regulartization term by lambda_?"
    assert np.isclose(J, 27), f"Wrong value. Expected {27}, got {J}. Check the regularization term"
    
    
    X_r = np.ones((num_movies_r, num_features_r))
    W_r = np.ones((num_users_r, num_features_r))
    b_r = np.ones((1, num_users_r))
    Y_r = np.ones((num_movies_r, num_users_r))
    R_r = np.ones((num_movies_r, num_users_r))

    # Evaluate cost function
    J = target(X_r, W_r, b_r, Y_r, R_r, 0);
    
    assert np.isclose(J, 90), f"Wrong value. Expected {90}, got {J}. Check the term without the regularization"
    
    
    X_r = np.ones((num_movies_r, num_features_r))
    W_r = np.ones((num_users_r, num_features_r))
    b_r = np.ones((1, num_users_r))
    Y_r = np.zeros((num_movies_r, num_users_r))
    R_r = np.ones((num_movies_r, num_users_r))

    # Evaluate cost function
    J = target(X_r, W_r, b_r, Y_r, R_r, 0);
    
    assert np.isclose(J, 160), f"Wrong value. Expected {160}, got {J}. Check the term without the regularization"
    
    X_r = np.ones((num_movies_r, num_features_r))
    W_r = np.ones((num_users_r, num_features_r))
    b_r = np.ones((1, num_users_r))
    Y_r = np.ones((num_movies_r, num_users_r))
    R_r = np.ones((num_movies_r, num_users_r))

    # Evaluate cost function
    J = target(X_r, W_r, b_r, Y_r, R_r, 1);
    
    assert np.isclose(J, 103.5), f"Wrong value. Expected {103.5}, got {J}. Check the term without the regularization"
    
    num_users_r = 3
    num_movies_r = 4 
    num_features_r = 4
    
    #np.random.seed(247)
    X_r = np.array([[0.36618032, 0.9075415,  0.8310605,  0.08590986],
                     [0.62634721, 0.38234325, 0.85624346, 0.55183039],
                     [0.77458727, 0.35704147, 0.31003294, 0.20100006],
                     [0.34420469, 0.46103436, 0.88638208, 0.36175401]])#np.random.rand(num_movies_r, num_features_r)
    W_r = np.array([[0.04786854, 0.61504665, 0.06633146, 0.38298908], 
                    [0.16515965, 0.22320207, 0.89826005, 0.14373251], 
                    [0.1274051 , 0.22757303, 0.96865613, 0.70741111]])#np.random.rand(num_users_r, num_features_r)
    b_r = np.array([[0.14246472, 0.30110933, 0.56141144]])#np.random.rand(1, num_users_r)
    Y_r = np.array([[0.20651685, 0.60767914, 0.86344527], 
                    [0.82665019, 0.00944765, 0.4376798 ], 
                    [0.81623732, 0.26776794, 0.03757507], 
                    [0.37232161, 0.19890823, 0.13026598]])#np.random.rand(num_movies_r, num_users_r)
    R_r = np.array([[1, 0, 1], [1, 0, 0], [1, 0, 0], [0, 1, 0]])#(np.random.rand(num_movies_r, num_users_r) > 0.4) * 1

    # Evaluate cost function
    J = target(X_r, W_r, b_r, Y_r, R_r, 3);
    
    assert np.isclose(J, 13.621929978531858, atol=1e-8), f"Wrong value. Expected {13.621929978531858}, got {J}."
    
    print('\033[92mAll tests passed!')


## numpy & matrix

In [3]:
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443
Average rating for movie 1 : 3.400 / 5



$$
\begin{aligned}
J(\mathbf{x}, \mathbf{w}, b) = \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \quad & \\
+ \underbrace{\frac{\lambda}{2} \sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2 + \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2}_{\text{regularization}}
\end{aligned}
$$

In [4]:
def cofi_cost_func(X, W, b, Y, R, lambda_) -> NDArray:
    z = (X @ W.T + b - Y) * R
    J = 0.5 * np.sum(z**2)
    std = lambda_ * 0.5 * (np.sum(np.square(W)) + np.sum(np.square(X))) 
    return J + std

def cofi_grad_func(X, W, b, Y, R, lambda_) -> Tuple[NDArray, NDArray, NDArray]:
    # 1. 计算误差矩阵 E
    # E 的形状是 (num_movies, num_users)
    E = ((X @ W.T + b) - Y) * R

    # 2. 计算对 X 的梯度
    # (E @ W) 计算无正则化的梯度，形状为 (num_movies, num_features)
    dX = (E @ W) + lambda_ * X

    # 3. 计算对 W 的梯度
    # (E.T @ X) 计算无正则化的梯度，形状为 (num_users, num_features)
    dW = (E.T @ X) + lambda_ * W

    # 4. 计算对 b 的梯度
    # np.sum(E, axis=0) 对每一列（每个用户）的误差求和
    # keepdims=True 确保结果的形状是 (1, num_users)，与 b 保持一致
    db = np.sum(E, axis=0, keepdims=True)

    return dX, dW, db

test_cofi_cost_func(cofi_cost_func)

All tests passed!


In [5]:
num_users_r = 1
num_movies_r = 1
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

print(f'X_r {X_r[0]}')
print(f'W_r {W_r}')
print(f'b_r {b_r}')
# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0.2);
print(f"Cost (with regularization): {J:0.2f}")

X_r [-0.03328682  1.1667464  -0.5064895 ]
W_r [[0.47935075 0.4766997  0.48794055]]
b_r [[0.23609531]]
Cost: 0.00
Cost (with regularization): 0.23


In [6]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(len(movieList))          #  Initialize my ratings

# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


In [7]:
Y, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

def normalize_ratings(Y, R) -> Tuple[NDArray, NDArray]:
    y_mean = (np.sum(Y*R,axis=1)/(np.sum(R, axis=1)+1e-12)).reshape(-1,1)
    y_norm = Y - np.multiply(y_mean, R) 
    # y_norm = Y - y_mean * R
    # y_norm = (Y - y_mean) * R
    return y_mean, y_norm

# Normalize the Dataset
y_norm, y_mean = normalize_ratings(Y, R)

In [8]:
class EarlyStopping:
    def __init__(self, patience, min_delta, X, W, b):
        self.patience = patience
        self.min_delta = min_delta
        
        # 内部状态
        self.counter = 0
        self.best_cost = np.inf
        self.best_X = X
        self.best_W = W     
        self.best_b = b
    
    def save_best(self, cost, X, W, b):
        self.best_cost = cost
        self.best_X = X
        self.best_W = W
        self.best_b = b

    def step(self, current_cost, X, W, b) -> bool:
        delta = np.abs(self.best_cost - current_cost)
        self.save_best(current_cost, X, W, b)
        if delta < self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                return True
            else:
                return False

        self.counter = 0

        return False

    def best_weights(self) -> Tuple[NDArray, NDArray, NDArray]:
        return self.best_X, self.best_W, self.best_b

In [9]:
num_movies, num_users = Y.shape
num_features = 100

W = np.random.normal(size=(num_users,  num_features))
X = np.random.normal(size=(num_movies, num_features))
b = np.random.normal(size=(1,          num_users))

# Instantiate an optimizer.
# optimizer = keras.optimizers.Adam(learning_rate=1e-1)


iterations = 10000
lambda_ = 1e-02
lr = 1e-01
print_interval = 20
cost = 0

patience = 10
min_delta = 1e-02
es = EarlyStopping(patience, min_delta, X, W, b)

beta1 = 0.9         # 一阶矩估计的指数衰减率
beta2 = 0.999       # 二阶矩估计的指数衰减率
epsilon = 1e-8      # 防止除以0的极小值
mX, vX = np.zeros_like(X), np.zeros_like(X)
mW, vW = np.zeros_like(W), np.zeros_like(W)
mb, vb = np.zeros_like(b), np.zeros_like(b)

start = time.time()
for i in range(iterations):
    cost = cofi_cost_func(X, W, b, y_norm, R, lambda_)
    if es.step(cost, X, W, b):
        X, W, b = es.best_weights()
        break
    dX, dW, db = cofi_grad_func(X, W, b, y_norm, R, lambda_)
    # --- 对 X 进行 Adam 更新 ---
    mX = beta1 * mX + (1 - beta1) * dX
    vX = beta2 * vX + (1 - beta2) * (dX ** 2)
    m_hatX = mX / (1 - beta1 ** (i + 1)) 
    v_hatX = vX / (1 - beta2 ** (i + 1))
    X -= lr * m_hatX / (np.sqrt(v_hatX) + epsilon)
    
    # --- 对 W 进行 Adam 更新 ---
    mW = beta1 * mW + (1 - beta1) * dW
    vW = beta2 * vW + (1 - beta2) * (dW ** 2)
    m_hatW = mW / (1 - beta1 ** (i + 1))
    v_hatW = vW / (1 - beta2 ** (i + 1))
    W -= lr * m_hatW / (np.sqrt(v_hatW) + epsilon)
    
    # --- 对 b 进行 Adam 更新 ---
    mb = beta1 * mb + (1 - beta1) * db
    vb = beta2 * vb + (1 - beta2) * (db ** 2)
    m_hatb = mb / (1 - beta1 ** (i + 1))
    v_hatb = vb / (1 - beta2 ** (i + 1))
    b -= lr * m_hatb / (np.sqrt(v_hatb) + epsilon)
    if (i+1) % print_interval == 0:
        # print(f'X {X[0,0]:.2f}, W {W[0,0]:.2f}, b {b[0,0]:.2f}, cost {cost:.2f}')
        t = time.time()
        print(f'Iteration {i+1:4d}: Cost {cost:,.8f} ({t-start:.8f} secs)') 

end = time.time()
print(f'Final cost: {cost:,.8f}')
print(f'Time: {end-start:.8f} seconds')

Iteration   20: Cost 34,938.78978500 (1.43009472 secs)
Iteration   40: Cost 6,315.52997317 (2.79775882 secs)
Iteration   60: Cost 3,196.17528220 (4.38217974 secs)
Iteration   80: Cost 2,662.47288345 (5.75847197 secs)
Iteration  100: Cost 2,472.78023983 (7.11897874 secs)
Iteration  120: Cost 2,341.17613259 (8.51323485 secs)
Iteration  140: Cost 2,225.30738346 (9.82243896 secs)
Iteration  160: Cost 2,117.86499321 (11.13273191 secs)
Iteration  180: Cost 2,016.98601912 (12.43845773 secs)
Iteration  200: Cost 1,921.90459935 (13.81118083 secs)
Iteration  220: Cost 1,832.15200881 (15.12111187 secs)
Iteration  240: Cost 1,747.36845122 (16.44136691 secs)
Iteration  260: Cost 1,667.24301768 (17.77261591 secs)
Iteration  280: Cost 1,591.49149048 (19.07895398 secs)
Iteration  300: Cost 1,519.84801797 (20.38658190 secs)
Iteration  320: Cost 1,452.06214414 (21.71529484 secs)
Iteration  340: Cost 1,387.89779244 (23.03955793 secs)
Iteration  360: Cost 1,327.13283978 (24.45054698 secs)
Iteration  380: 

In [ ]:
p = X @ W.T + b

#restore the mean
pm = p + y_mean

my_predictions = pm[:,0]

# sort predictions
top_n = 10
ix = np.argsort(my_predictions)[top_n:][::-1]

for i in range(top_n):
    j = ix[i]
    print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 5.00 for movie Shrek (2001)
Predicting rating 5.00 for movie Lord of the Rings: The Return of the King, The (2003)
Predicting rating 5.00 for movie Harry Potter and the Chamber of Secrets (2002)
Predicting rating 5.00 for movie Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Predicting rating 5.00 for movie Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Predicting rating 5.00 for movie Incredibles, The (2004)
Predicting rating 5.00 for movie Toy Story 3 (2010)
Predicting rating 4.12 for movie Dark Knight, The (2008)
Predicting rating 4.08 for movie Lord of the Rings: The Fellowship of the Ring, The (2001)
Predicting rating 4.04 for movie Memento (2000)


Original vs Predicted ratings:

Original 5.0, Predicted 5.00 for Shrek (2001)
Original 5.0, Predicted 5.00 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.00 for Amelie (Fabuleux d

In [173]:
filter=(movieList_df["number of ratings"] > 100)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("number of ratings", ascending=False)

,pred,mean rating,number of ratings,title
393,4.082643,4.106061,198,"Lord of the Rings: The Fellowship of the Ring,..."
653,4.037525,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
929,5.000226,4.118919,185,"Lord of the Rings: The Return of the King, The..."
246,5.000446,3.867647,170,Shrek (2001)
51,3.685657,3.938235,170,Gladiator (2000)
211,4.043878,4.122642,159,Memento (2000)
793,4.999433,3.778523,149,Pirates of the Caribbean: The Curse of the Bla...
2112,4.121277,4.238255,149,"Dark Knight, The (2008)"
773,3.798777,3.960993,141,Finding Nemo (2003)
79,3.542542,3.699248,133,X-Men (2000)
